
### 1. What is Lazy Evaluation?

> Spark doesn't execute transformations immediately. It builds a logical execution plan and only executes it when an action is called. This is called **lazy evaluation**. It allows Spark to optimize the complete execution plan before running it.

**Example:**

python
df2 = df.filter("amount > 1000")
df3 = df2.select("customer_id", "amount")
# Nothing is actually executed yet
df3.count()  # triggers execution


**Easy interview line:**  
*Transformations build the plan; actions trigger execution.*

---

### 2. Transformation vs Action

- **Transformation:** Creates a new DataFrame/RDD from an existing one.  
  Examples: `filter()`, `select()`, `withColumn()`, `join()`, `groupBy()`
- **Action:** Triggers Spark execution and returns a result or writes data.  
  Examples: `count()`, `show()`, `collect()`, `write`

> Interview answer:  
*Transformations are lazily evaluated, while actions trigger the actual Spark job.*

---

### 3. Narrow vs Wide Transformation

- **Narrow Transformation:** Each output partition depends on a small number of input partitions, usually without shuffle.  
  Examples: `filter`, `select`, `map`
- **Wide Transformation:** Data needs to move between partitions, causing a shuffle.  
  Examples: `groupBy`, `join`, `distinct`, `orderBy`, `repartition`

**Simple:**


Narrow: Partition 1 → Partition 1
 
Wide: Partition 1 ─┐
       Partition 2 ─┼→ Shuffle → New partitions
       Partition 3 ─┘

> Interview line:  
*Wide transformations usually involve shuffle, which can be expensive because data moves across executors.*

---

### 4. What causes a Shuffle?

> Shuffle happens when Spark needs to redistribute data across partitions, usually based on a key.

Common examples:  
- `groupBy()`
- `join()`
- `distinct()`
- `orderBy()`
- `repartition()`

**Example:**

python
df.groupBy("customer_id").sum("amount")


---

### 5. What is Data Skew?

> Data skew occurs when data is unevenly distributed across partitions. For example, if one customer has 50 million records, while others have a few hundred, the partition containing that customer can become much larger and slower.


Partition 1 → 10K records
Partition 2 → 12K records
Partition 3 → 15K records
Partition 4 → 50 MILLION records  ← Skew

*The entire stage may wait for that one slow partition.*

---

### 6. How do you identify Data Skew?

> "I first look at the Spark UI, particularly the stage details. If one or a few tasks have significantly higher input size, shuffle read, shuffle write or execution time than the other tasks, that is a strong indication of skew."

**Example:**

python
df.groupBy("customer_id").count().orderBy("count", ascending=False).show()

If you see:

customer_id | count
101         | 50,000,000
102         | 100
103         | 200

That's a clear skew candidate.

---

### 7. How do you solve Data Skew?

> Several techniques depending on the use case:

1. **Broadcast join**  
   If one side is small:
   python
   from pyspark.sql.functions import broadcast
 
   df_large.join(
       broadcast(df_small),
       "customer_id"
   )
   
2. **Salting**  
   Add a random/salt value to distribute skewed keys.
3. **Adaptive Query Execution**  
   AQE can mitigate certain skewed shuffle partitions with skew join optimization.
4. **Repartitioning**  
   Redistribute data appropriately.

> Interview answer:  
*"I first confirm skew using Spark UI. Then depending on the scenario, I might use broadcast joins, salting, repartitioning or AQE skew-join optimization."*

---

### 8. What is Broadcast Join?

> "A broadcast join copies the smaller DataFrame to each executor so Spark doesn't need to shuffle the large DataFrame for the join."

**Example:**

python
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "customer_id"
)


Conceptually:

Small table
    ↓
Broadcast
 ↙  ↓  ↘
E1  E2  E3

Large table stays distributed


---

### 9. When should you use Broadcast Join?

> "I use it when one side of the join is small enough to fit comfortably in executor memory."

**Example:**
- Transactions → 500 GB
- Customer → 50 MB
- Broadcasting `Customer` can be appropriate.

> Don't say: *Always broadcast the smaller table.*  
> Better: *Broadcast when the smaller side is sufficiently small and safe for executor memory.*

---

### 10. What happens if the Broadcast table is too large?

> "The broadcast data has to be distributed to executors and held in memory. If it's too large, it can cause high memory usage, executor OOM, or performance degradation. In such cases, avoid forcing broadcast join and consider a normal distributed join or another optimization."

---

### 11. repartition() vs coalesce()

#### repartition()
> Changes the number of partitions using a shuffle. It can increase or decrease partitions.
python
df = df.repartition(20)

#### coalesce()
> Mainly used to reduce the number of partitions without a full shuffle.
python
df = df.coalesce(5)

**Easy memory trick:**  
- `repartition` → shuffle → more flexible  
- `coalesce`    → reduce partitions → less shuffle

> Interview Q:
- "When would you use repartition?"  
  *Before a large operation or when a specific partitioning strategy is needed.*
- "When coalesce?"  
  *Before writing a smaller output, and don't need a full redistribution.*

---

### 12. How do you optimize joins?

> My answer would be:

*"First I check the join size and data distribution. I filter and select only required columns before the join, broadcast the smaller side when appropriate, avoid unnecessary joins and check for data skew. I also look at the Spark UI and query plan to understand whether a large shuffle is happening."*

**Example:**
python
small = (
    customer_df
    .select("customer_id", "segment")
)

result = transaction_df.join(
    broadcast(small),
    "customer_id"
)


---

### 13. How do you reduce Shuffle?

> "I try to reduce unnecessary wide transformations. I filter data early, select only required columns, use broadcast joins for small datasets, avoid unnecessary repartitioning, handle skew and choose appropriate partitioning."

**Example:**

Instead of:
python
df.join(huge_lookup, "id")  # after carrying 50 columns

Do:
python
lookup = huge_lookup.select("id", "category")
df = df.filter("amount > 0")
df.join(lookup, "id")

*Less data needs to move.*

---

### 14. What is Caching?

> "Caching stores a DataFrame's computed data so that if the same DataFrame is used multiple times, Spark can avoid recomputing it from the source."

**Example:**

python
df.cache()
df.count()
df.groupBy("customer_id").count().show()


---

### 15. cache() vs persist()

> "cache() uses Spark's default storage level, while persist() allows you to specify the storage level."

**Example:**
python
df.cache()
from pyspark import StorageLevel
df.persist(StorageLevel.MEMORY_AND_DISK)

> Interview answer:  
*"I use caching only when the DataFrame is reused and recomputation is expensive. I don't cache every DataFrame because unnecessary caching consumes memory."*

---

### 16. What is Adaptive Query Execution?

> "AQE allows Spark to optimize the query during execution using runtime statistics. It can dynamically adjust things such as shuffle partition handling, join strategies and skewed partitions."

**Common AQE capabilities:**
- Coalescing shuffle partitions
- Dynamic join strategy optimization
- Skew join optimization

> Interview answer:  
*"AQE is useful because Spark can make better decisions after seeing the actual runtime data instead of relying only on estimates made before execution."*

---

### 17. What is Spark UI and how do you use it?

> "Spark UI is used to understand how a Spark application is executing and identify performance bottlenecks."

**I mainly check:**
- **Jobs:** Running jobs and their durations
- **Stages:** Where expensive operations occur
- **Tasks:** Whether some tasks are significantly slower
- **SQL:** Physical execution plan & SQL operations
- **Executors:** Memory, CPU, task failures
- **Storage:** Cached data and storage usage

---

### 18. How do you troubleshoot a slow Spark job?

> Answer systematically:

*"First, I compare the current execution time with the previous successful run. Then I open Spark UI and identify the slow job and stage. I check for large shuffles, data skew, long-running tasks, excessive partitions, spills and executor issues. Then I inspect the query plan and joins. I also check input data volume and file sizes because the data volume may have increased. Finally, I check cluster resources and configuration."*

**Troubleshooting flow:**


Slow Job
   ↓
Spark UI
   ↓
Jobs
   ↓
Stages
   ↓
Tasks
   ↓
Shuffle / Skew / Spill
   ↓
Partitions
   ↓
Joins
   ↓
File sizes
   ↓
Query plan
   ↓
Cluster resources


---

### 19. What causes the Small File Problem?

> "The small-file problem happens when a table contains a very large number of small files. Spark spends time listing, opening and managing files, instead of processing data."

**Common causes:**
- Too many small streaming batches
- Excessive partitions
- Frequent small writes
- Poor ingestion design
- Over-partitioning

**Example:**

Bad:

1 million files × 1 MB

Better:

10,000 files × 100 MB

*Principle: avoid excessive tiny files.*

---

### 20. How do you optimize a Delta table?

> "First, I look at the workload and file layout. If there are many small files, I use OPTIMIZE to compact them. For selective queries, consider ZORDER. Review partitioning to avoid excess partitions. Use VACUUM carefully to remove old files."

**Simple:**

Delta Optimization
       ↓
Small files → OPTIMIZE
       ↓
Selective queries → ZORDER (where appropriate)
       ↓
Partitioning → Review
       ↓
Old files → VACUUM carefully


---

### 🔥 Common Scenarios & Interview Q&A

#### "Your Databricks job was taking 20 minutes earlier, but now it takes 2 hours. How will you troubleshoot?"

> Don't immediately say: *Increase the cluster.*  
> Structured approach:

*"First, I would compare the current run with the previous run to understand what changed. Then I would open Spark UI and identify which job/stage is taking most time. I would check if input data volume increased, presence of large shuffle, data skew, excessive partitions, spills or long-running tasks. Then I would inspect joins and query plan, check number/size of input files for a small-file problem, and check executor CPU/memory and cluster configuration. Optimize bottleneck, don't blindly increase cluster size."*

---

#### "Walk me through Spark UI."

> "I go to Jobs and identify the slow job. Then open the relevant stage and compare task execution times. If one or a few tasks are much slower, check data skew. If shuffle read/write is high, check joins, aggregations or repartitioning. Also check spill metrics, input/output size, executor utilization. Correlate with SQL/physical plan."

---

#### "You found a huge shuffle. What do you do?"

> "Identify what's causing the shuffle—join, groupBy, distinct, orderBy or repartition. Then filter early, select fewer columns, broadcast small table, reduce unnecessary repartitioning, address data skew."

---

#### "One task takes 40 minutes, others take 2 minutes."

> "That's uneven partition distribution—most likely data skew. Inspect key distribution, check if few keys hold disproportionate data. Use broadcast joins, salting, repartitioning or AQE skew handling depending on case."

---

#### "Input data increased from 100 GB to 1 TB."

> "Increase in processing time may be expected, but check if pipeline scales efficiently—partitioning, file sizes, parallelism, joins, shuffle, cluster resources. Only then consider scaling the cluster."

---

#### "Cluster CPU is only 20%, but job is very slow."

> "Low CPU can indicate job is waiting on I/O, shuffle, skewed tasks, serialization, or other bottlenecks. Inspect Spark UI and stage/task metrics before increasing cluster size."

---

#### "One join became very slow after data increased."

> "Check join strategy and dataset size. If one side still small, use broadcast join. If both large, check shuffle size, partitioning, data skew, and if query plan changed."

---

#### ⭐ Performance Troubleshooting Framework

> When you hear: `"Job became slow."`  
> Think in this order:

1. Did data volume increase?
          ↓
2. Spark UI
          ↓
3. Which stage is slow?
          ↓
4. Shuffle?
          ↓
5. Data skew?
          ↓
6. Too many / too few partitions?
          ↓
7. Join strategy?
          ↓
8. Small files?
          ↓
9. Query/physical plan?
          ↓
10. CPU / memory / spill / cluster resources?

---

> Final answer:  
*"I first identify the bottleneck using Spark UI, then optimize the specific issue—rather than immediately increasing cluster size."*  
This demonstrates strong performance troubleshooting, not just Spark terminology.

---